In [23]:
# Imports and setup
import os
import json
from collections import defaultdict, Counter
from pathlib import Path
from tqdm import tqdm

DATASET_BASE = "/workspace/dataset"
COLLECTIONS = ["lastfm", "suno", "udio"]
OUTPUT_DIR = "/workspace/ngram_datasets"

print("Imports loaded")

Imports loaded


In [24]:
# --- Core functions ---

def parse_functional_lab(path: str) -> list:
    """Parse a _functional.lab file and return the list of functional labels (Roman numerals)."""
    labels = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 3:
                continue
            label = parts[2]
            if label == "N":
                continue
            labels.append(label)
    return labels


def make_ngram(chords, n=3):
    """
    Create n-grams from chord sequence.
    - Remove consecutive duplicates first.
    - Trigrams (n=3): All 3 chords must be different (case-insensitive).
    - Tetragrams (n=4+): At least 3 different chords required.
    """
    if len(chords) < n:
        return []

    # Remove consecutive duplicates
    filtered = [chords[0]]
    for c in chords[1:]:
        if c != filtered[-1]:
            filtered.append(c)

    ngrams = []
    for i in range(len(filtered) - n + 1):
        ngram = tuple(filtered[i:i + n])
        unique = len(set(c.lower() for c in ngram))
        if n == 3 and unique == 3:
            ngrams.append(ngram)
        elif n != 3 and unique >= 3:
            ngrams.append(ngram)
    return ngrams


# --- Genre normalization ---
target_genres = [
    'rock', 'pop', 'jazz', 'electronic', 'blues', 'metal', 'hiphop', 'country',
    'folk', 'soul', 'punk', 'classical', 'reggae', 'rnb', 'indie', 'funk',
    'dance', 'latin', 'ambient', 'experimental', 'world'
]

explicit_map = {
    'pink': 'pop', 'p!nk': 'pop', 'beatles': 'rock', 'the beatles': 'rock',
    'stones': 'rock', 'the stones': 'rock', 'alternative': 'rock', 'emo': 'punk',
    'shoegaze': 'rock', 'acoustic': 'folk', 'instrumental': 'other', 'soundtrack': 'other',
    'british': 'pop', '80s': 'pop', "80's": 'pop', '90s': 'rock', '90': 'rock',
    '70s': 'pop', '70': 'pop', 'raphiphop': 'hiphop', 'hip hop': 'hiphop', 'rap': 'hiphop',
    'indie rock': 'rock', 'indie': 'indie', 'funksoulrnb': 'soul', 'folkcountry': 'country',
}

def normalize_tag(tag):
    if tag is None:
        return 'other'
    t = tag.strip().lower()
    if t in explicit_map:
        return explicit_map[t]
    if t in target_genres:
        return t
    for genre in target_genres:
        if genre in t:
            return genre
    return 'other'


def extract_styles(style_json_path):
    """Extract and normalize styles from style JSON file."""
    with open(style_json_path, "r") as f:
        data = json.load(f)

    original = data.get("original", None)
    normalized_original_styles = []
    if isinstance(original, list):
        for style in original:
            ns = normalize_tag(style)
            if ns != 'other':
                normalized_original_styles.append(ns)
    elif isinstance(original, str):
        for style in [s.strip() for s in original.split(",") if s.strip()]:
            ns = normalize_tag(style)
            if ns != 'other':
                normalized_original_styles.append(ns)

    genres = data.get("genre_dortmund", {})
    dortmund_genres = [{'genre': normalize_tag(g), 'confidence': float(c)} for g, c in genres.items()]
    return normalized_original_styles, dortmund_genres


print("Core functions defined")

Core functions defined


In [25]:
# --- Build n-gram dataset from _functional.lab files ---

def create_ngram_dataset(base_path=DATASET_BASE, collections=COLLECTIONS, n=3):
    """
    Build n-gram aggregated dataset by reading pre-computed _functional.lab files.
    No tonality estimation or Roman numeral conversion — just read and form n-grams.
    """
    collection_data = {}

    for collection in collections:
        print(f"\nProcessing: {collection}")
        collection_path = os.path.join(base_path, collection)
        if not os.path.exists(collection_path):
            print(f"  Warning: {collection_path} does not exist, skipping")
            continue

        ngram_data = defaultdict(lambda: {
            "count": 0,
            "human_styles": Counter(),
            "dortmund_genres": Counter(),
            "song_ids": []
        })

        song_dirs = [d for d in os.listdir(collection_path)
                     if os.path.isdir(os.path.join(collection_path, d))]

        processed = 0
        skipped_no_file = 0
        skipped_empty = 0

        for song_id in tqdm(song_dirs, desc=f"  {collection}"):
            song_path = os.path.join(collection_path, song_id)
            func_lab = os.path.join(song_path, f"{song_id}_functional.lab")

            if not os.path.exists(func_lab):
                skipped_no_file += 1
                continue

            labels = parse_functional_lab(func_lab)
            if not labels:
                skipped_empty += 1
                continue

            ngrams = make_ngram(labels, n)
            if not ngrams:
                skipped_empty += 1
                continue

            processed += 1

            # Load styles (optional)
            style_json = os.path.join(song_path, f"{song_id}_style.json")
            original_styles, dortmund_genres = [], []
            if os.path.exists(style_json):
                try:
                    original_styles, dortmund_genres = extract_styles(style_json)
                except Exception:
                    pass

            for ngram in ngrams:
                key = tuple(ngram)
                ngram_data[key]["count"] += 1
                if song_id not in ngram_data[key]["song_ids"]:
                    ngram_data[key]["song_ids"].append(song_id)
                for style in original_styles:
                    ngram_data[key]["human_styles"][style] += 1
                for gi in dortmund_genres:
                    ngram_data[key]["dortmund_genres"][gi['genre']] += gi['confidence']

        collection_data[collection] = dict(ngram_data)
        print(f"  {len(ngram_data):,} unique {n}-grams | {processed:,} songs processed | "
              f"{skipped_no_file:,} no functional.lab | {skipped_empty:,} empty/no ngrams")

    return collection_data


print("Dataset builder defined")

Dataset builder defined


In [26]:
# --- Export function ---

def export_ngram_dataset(collection_data, n, output_dir=OUTPUT_DIR):
    """Export n-gram dataset to JSON files (same format as before)."""
    subdir = {3: "trigrams", 4: "tetragrams"}.get(n, f"{n}grams")
    output_path = os.path.join(output_dir, subdir)
    os.makedirs(output_path, exist_ok=True)

    for collection, ngram_data in collection_data.items():
        export_data = []
        for ngram_tuple, stats in ngram_data.items():
            export_data.append({
                "ngram": list(ngram_tuple),
                "count": stats["count"],
                "human_styles": stats["human_styles"].most_common(10),
                "dortmund_genres": sorted(stats["dortmund_genres"].items(), key=lambda x: x[1], reverse=True),
                "song_ids": stats["song_ids"],
                "num_songs": len(stats["song_ids"])
            })
        export_data.sort(key=lambda x: x["count"], reverse=True)

        output_file = os.path.join(output_path, f"{n}gram_dataset_{collection}_ace.json")
        with open(output_file, 'w') as f:
            json.dump(export_data, f, indent=2)
        print(f"Exported {len(export_data):,} {n}-grams for {collection} → {output_file}")


print("Export function defined")

Export function defined


In [27]:
# --- Run: Create and export trigram datasets ---

trigram_data = create_ngram_dataset(DATASET_BASE, COLLECTIONS, n=3)

print("\n" + "=" * 80)
print("EXPORTING TRIGRAM DATASETS")
print("=" * 80)
export_ngram_dataset(trigram_data, n=3)

# Summary
print("\n" + "=" * 80)
print("TRIGRAM STATISTICS")
print("=" * 80)
for collection, ngram_data in trigram_data.items():
    total = sum(s["count"] for s in ngram_data.values())
    unique = len(ngram_data)
    print(f"\n{collection.upper()}:")
    print(f"  Unique trigrams: {unique:,}")
    print(f"  Total instances: {total:,}")
    if ngram_data:
        top_key = max(ngram_data, key=lambda k: ngram_data[k]["count"])
        top = ngram_data[top_key]
        print(f"  Most common: {' - '.join(top_key)} (count: {top['count']:,})")


Processing: lastfm


  lastfm: 100%|██████████| 19913/19913 [00:15<00:00, 1280.76it/s]


  28,573 unique 3-grams | 18,448 songs processed | 997 no functional.lab | 468 empty/no ngrams

Processing: suno


  suno: 100%|██████████| 19972/19972 [00:23<00:00, 835.39it/s]


  16,920 unique 3-grams | 18,547 songs processed | 994 no functional.lab | 431 empty/no ngrams

Processing: udio


  udio: 100%|██████████| 19992/19992 [00:10<00:00, 1898.96it/s]


  24,893 unique 3-grams | 18,210 songs processed | 961 no functional.lab | 821 empty/no ngrams

EXPORTING TRIGRAM DATASETS
Exported 28,573 3-grams for lastfm → /workspace/ngram_datasets/trigrams/3gram_dataset_lastfm_ace.json
Exported 16,920 3-grams for suno → /workspace/ngram_datasets/trigrams/3gram_dataset_suno_ace.json
Exported 24,893 3-grams for udio → /workspace/ngram_datasets/trigrams/3gram_dataset_udio_ace.json

TRIGRAM STATISTICS

LASTFM:
  Unique trigrams: 28,573
  Total instances: 859,882
  Most common: IV - I - V (count: 11,194)

SUNO:
  Unique trigrams: 16,920
  Total instances: 1,052,255
  Most common: IV - I - V (count: 28,093)

UDIO:
  Unique trigrams: 24,893
  Total instances: 608,663
  Most common: IV - I - V (count: 8,136)
